### A5.1.14. Fix Data Race in Class Methods

**Problem:**

Given a class with shared mutable state accessed from multiple threads, identify the data race and fix it by adding proper synchronization.

**Explanation:**

A data race occurs when two threads access the same variable concurrently and at least one of them writes. The result is unpredictable. The fix is to protect shared state with a lock so only one thread can read or write at a time.

How to fix it:

1. Identify the shared mutable state (e.g., a counter, a list, a dictionary).
2. Identify which methods read and write that state.
3. Create a `threading.Lock`.
4. Wrap every access (read or write) to the shared state in `with self.lock:` so only one thread operates on it at a time.

**Example:**

A `BankAccount` class has `deposit` and `withdraw` called from multiple threads. Without a lock, the balance can become incorrect. Adding a lock around balance updates fixes the race.

In [ ]:
import threading


class BankAccountBroken:
    def __init__(self, balance):
        self.balance = balance

    def deposit(self, amount):
        current = self.balance
        self.balance = current + amount

    def withdraw(self, amount):
        current = self.balance
        self.balance = current - amount


class BankAccountFixed:
    def __init__(self, balance):
        self.balance = balance
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            self.balance += amount

    def withdraw(self, amount):
        with self.lock:
            self.balance -= amount


def stress_test(account_class):
    account = account_class(1000)

    def deposit_many():
        for _ in range(10000):
            account.deposit(1)

    def withdraw_many():
        for _ in range(10000):
            account.withdraw(1)

    threads = [
        threading.Thread(target=deposit_many),
        threading.Thread(target=withdraw_many),
    ]

    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()

    return account.balance


broken_result = stress_test(BankAccountBroken)
fixed_result = stress_test(BankAccountFixed)

print(f"Broken account balance (expected 1000): {broken_result}")
print(f"Fixed account balance (expected 1000): {fixed_result}")

**References:**

📘 Python Documentation — [threading.Lock](https://docs.python.org/3/library/threading.html#lock-objects)

📘 Herlihy, M. & Shavit, N. (2012). *The Art of Multiprocessor Programming* — Data Races

---

[⬅️ Previous: Build PIMPL Wrapper Class](./13_build_pimpl_wrapper_class.ipynb) | [Next: Reduce Lock Contention in Class Design ➡️](./15_reduce_lock_contention_in_class_design.ipynb)